# Cleaning TESS images
Below is some example code for downloading all images from one TESS frame and cleaning them
using the approach described in Alexandersen et al. 2026.

To run this notebook, you will need a Python environment with the neccessary dependencies. 
This can be set up like so:
```sh
conda create -n cleany python=3.12 numpy astropy jupyter notebook astroquery ccdproc sep
conda activate cleany
```

### Setup

In [1]:
# Third party imports
import os
import sys
import copy
import glob
import requests
import warnings
import matplotlib
import subprocess
import numpy as np
from pathlib import Path
from importlib import reload
import matplotlib.pyplot as plt
from astropy.visualization import ZScaleInterval
warnings.filterwarnings("ignore")

In [2]:
# Plot setup
matplotlib.rc("font", size=22, family="serif", weight="bold")
plt.rcParams["figure.figsize"] = [10, 4]
np.printoptions(suppress=True)

In [3]:
# Local imports
sys.path.append(str(Path.cwd().parent))
from cleany import imagehandler
from cleany import cleaner
reload(imagehandler)  # If you make changes to the background code, just rerun this cell to re-import, rather than killing the notebook
reload(cleaner)       # If you make changes to the background code, just rerun this cell to re-import, rather than killing the notebook

<module 'cleany.cleaner' from '/home/mikea/GitHub/cleany/cleany/cleaner.py'>

In [4]:
%%time
# Define a function that we can use to make some simple plots after each step.
def make_plots(input_data, title=""):
    """
    Make two plots of the same data.
    One has the colourbare going from the minimum to maximum data values,
    while the other uses the common ZScaleInterval to select an interval that shows
    the section of the dynamic range that most data values lie within.
    """
    fig, ax = plt.subplots(1, 2)
    # Masked pixels have np.nan values, which show up as white in imshow, which is very distracting.
    # To make the masked pixels less distracting, masked pixels will be plotted as the mean value. 
    image_data = copy.deepcopy(input_data)
    image_data[np.isnan(image_data)] = np.nanmean(image_data)
    # Left side, min to max
    im=ax[0].imshow(image_data, interpolation='nearest', cmap='gray', origin='upper',
                    vmin=np.nanmin(image_data), vmax=np.nanmax(image_data))
    fig.colorbar(im, ax=ax[0])
    ax[0].set_title("Min to max")
    # Right side, ZScaleInterval
    z1, z2 = ZScaleInterval().get_limits(image_data)
    im=ax[1].imshow(image_data, interpolation='nearest', cmap='gray', origin='upper', vmin=z1, vmax=z2)
    fig.colorbar(im, ax=ax[1])
    ax[1].set_title("ZScale")
    if title:
        fig.suptitle(title)

CPU times: user 3 μs, sys: 1 μs, total: 4 μs
Wall time: 5.96 μs


### Get the data

In [5]:
# Define which sector/camera/ccd to look at
sector = 5
camera = 1
ccd = 4
# Define the name of the download script.
download_script = f"tesscurl_sector_{sector}_ffic.sh"
# Define where to put good, bad and cleaned data:
used_data_dir = Path("good_quality")
used_data_dir.mkdir(exist_ok=True)
unused_data_dir = Path("poor_quality")
unused_data_dir.mkdir(exist_ok=True)
data_dir = Path("cleaned")
data_dir.mkdir(exist_ok=True)

# Here we're going to grab the download script for all of the sector,
# filtering it to only get the camera/ccd that we want. 
with requests.get(f"https://archive.stsci.edu/missions/tess/download_scripts/sector/{download_script}") as r:
    r.raise_for_status()
    with open(download_script, 'w') as f:
        for line in r.text.splitlines():
            if f"s0005-1-4" in line:
                print(line, file=f)

In [5]:
%%time
# Run the download script to download the images. 
# This will take a while, downloading ~11 GB of data
subprocess.run(["/bin/bash", download_script])
# This wil download 1196 fits images, so be patient.

** Resuming transfer from byte position 35547840
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 33.9M  100 33.9M    0     0  27.6M      0  0:00:01  0:00:01 --:--:-- 38.8M
** Resuming transfer from byte position 35547840
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0  

KeyboardInterrupt: 

In [73]:
%%time
# Split good and poor quality images.

# Originally we used Eleanor to filter the images and remove ones with high background or poor image quality,
# but here we will just use the list of known good images.
good_images = Path("good_image_names.txt").read_text().splitlines()
Path("poor_quality").mkdir(parents=True, exist_ok=True)
for f in Path(".").glob("tess*_ffic.fits"):
    print(f)
    if str(f) not in good_images:
        f.replace(f.parent / unused_data_dir / f.name)
        print(f"{f} moved to {(f.parent / unused_data_dir / f.name)}")
    else:
        f.replace(f.parent / used_data_dir / f.name)
        print(f"{f} moved to {(f.parent / used_data_dir / f.name)}")    

CPU times: user 1.38 ms, sys: 31 μs, total: 1.41 ms
Wall time: 915 μs


### Clean the data

In [ ]:
%%time
filenames = sorted(used_data_dir.glob("tess*_ffic.fits"))[0::]
print(len(filenames), "files")
C = cleaner.DataCleaner(filenames, extno=1, EXPTIME="EXPOSURE", EXPUNIT="d", MAGZERO=18.0,
                        MJD_START="BJDREFI+TSTART+-2400000.5", GAIN="GAINA", FILTER="-Tess", verbose=False,
                        xycuts=[44, 2092, 0, 2048]  # Removes overscan strips. Modify to use smaller area.
                        #xycuts=[44, 244, 0, 200]  # Just do a 200x200 cutout.
                        )
# Note: MAGZERO=18.0 is not accurate, but it also doesn't get used anywhere. When I made the method, I thought it would be...

# Do rough alignment (to pixel precision, without reprojection)
# TESS is usually already aligned to this level, but it's here to be sure, since the reprojection is faster if this is done first.
C.rough_align(0)
# Align by reprojecting/resampling the data onto the pixel grid of image 0 using the WCS of each image.
# (they'll thus have identical WCS afterwards and we can work in pixel space.)
C.reproject_data(0)
# Save this step (comment out if you just want the next step to re-use the C object)
C.save_cleaned(str(data_dir/"clean00"))
# Plot
make_plots(C.cleaned_data.data[0])

1108 files
Reading image 827: good_quality/tess2018339035938-s0005-1-4-0125-s_ffic.fits

In [ ]:
%%time
"""
Subtract the median background level from each images.
"""
# If you want to simply reuse the existing DataCleaner object, comment out the lines from here ...
filenames = sorted(data_dir.glob("clean00*.fits"))[0::]
print(len(filenames), "files")
C = cleaner.DataCleaner(filenames, extno=0, EXPTIME="EXPOSURE", EXPUNIT="d", MAGZERO=18.0,
                        MJD_START="BJDREFI+TSTART+-2400000.5", GAIN="GAINA", FILTER="-Tess", verbose=False)
# ... to here (you can also consider commenting out the `save_cleaned` step in the previous cell)

C.subtract_background_level(mode='median')
# Save this step (comment out if you just want the next step to re-use the C object)
C.save_cleaned(str(data_dir/"clean01"))
# Plot
make_plots(C.cleaned_data.data[220])

In [ ]:
%%time
"""
Mask bright pixels and print the fraction of pixels masked.
"""
# If you want to simply reuse the existing DataCleaner object, comment out the lines from here ...
filenames = sorted(data_dir.glob("clean01*.fits"))[0::]
print(len(filenames), "files")
C = cleaner.DataCleaner(filenames, extno=0, EXPTIME="EXPOSURE", EXPUNIT="d", MAGZERO=18.0,
                        MJD_START="BJDREFI+TSTART+-2400000.5", GAIN="GAINA", FILTER="-Tess", verbose=False)
# ... to here (you can also consider commenting out the `save_cleaned` step in the previous cell)

C.mask_bright_sources(20000, 3.0)  # Large mask around very bright sources
C.mask_bright_sources(2000, 1.0)  # Smaller mask around lesser bright sources
# Print fraction of pixels masked
print(np.sum(np.isnan(C.cleaned_data.data))/(np.prod(np.shape(C.cleaned_data.data))))
# Save this step
C.save_cleaned(str(data_dir/"clean02"))
# Plot
make_plots(C.cleaned_data.data[220])

In [ ]:
%%time
"""
Do the donut template subtraction.

Since there is a big time gap in the middle of the sector, the donut template should not straddle this gap. 
We thus do donut subtraction of the two halves separately.
We therefore have to read each half into separate cleaner objects, rather than using the inherited cleaner object from above.
"""
filenames = sorted(data_dir.glob("clean02*.fits"))[0:547:]
print(len(filenames), "files")
D1 = cleaner.DataCleaner(filenames, extno=0, EXPTIME="EXPOSURE", EXPUNIT="d", MAGZERO=18.0,
                        MJD_START="BJDREFI+TSTART+-2400000.5", GAIN="GAINA", FILTER="-Tess", verbose=False)
D1.subtract_background_level(mode='median')
D1.template_subtract('donut', usemean=False, ninner=4, nouter=10)  # ninner = h, nouter = h + d
D1.subtract_background_level(mode='sep')
D1.save_cleaned(str(data_dir/"clean03a"))

# Plot
make_plots(D1.cleaned_data.data[220])

# Second half
filenames = sorted(data_dir.glob("clean02*.fits"))[547::]
print(len(filenames), "files")
D2 = cleaner.DataCleaner(filenames, extno=0, EXPTIME="EXPOSURE", EXPUNIT="d", MAGZERO=18.0,
                        MJD_START="BJDREFI+TSTART+-2400000.5", GAIN="GAINA", FILTER="-Tess", verbose=False)
D2.subtract_background_level(mode='median')
D2.template_subtract('donut', usemean=False, ninner=4, nouter=10)
D2.subtract_background_level(mode='sep')
D2.save_cleaned(str(data_dir/"clean03b"))

# Plot
make_plots(D2.cleaned_data.data[220])

In [ ]:
"""
Do the final background subtraction using three separate methods (for comparison).

- Source Extractor Background Estimator.
- median
- mean
"""
filenames = sorted(data_dir.glob("clean03*.fits"))[0::]
print(len(filenames), "files")
F1 = cleaner.DataCleaner(filenames, extno=0, EXPTIME="EXPOSURE", EXPUNIT="d", MAGZERO=18.0,
                        MJD_START="BJDREFI+TSTART+-2400000.5", GAIN="GAINA", FILTER="-Tess", verbose=False)
F2=copy.deepcopy(F1)  # Much faster than re-reading the files
F3=copy.deepcopy(F1)  # Much faster than re-reading the files
# Final cleaning by subtracting a Source Extractor background estimate map from each frame.
F1.subtract_background_level(mode='sep')
F1.save_cleaned(str(data_dir/"clean04_sep"))
# Final cleaning by subtracting the median of each frame
F2.subtract_background_level(mode='median')
F2.save_cleaned(str(data_dir/"clean04_median"))
# Final cleaning by subtracting the mean of each frame
F3.subtract_background_level(mode='mean')
F3.save_cleaned(str(data_dir/"clean04_mean"))

# Plot
make_plots(F1.cleaned_data.data[220], title="sep")
make_plots(F2.cleaned_data.data[220], title="median")
make_plots(F3.cleaned_data.data[220], title="mean")